# 06 - Key insights

**Audience:** a reader without a background in air quality or machine learning.

**Question of the project:** when and where is Seoul's air most harmful, and can tomorrow's PM2.5 level be predicted well enough to warn residents?

PM2.5 is fine dust: particles small enough to reach deep into the lungs. Korea grades a day's average as Good (up to 15 ug/m3), Moderate (16-35), Bad (36-75) or Very bad (above 75). The data covers 25 monitoring stations, one per district of Seoul, hour by hour from 2017 to 2019.

Every number below is loaded from the files the earlier notebooks wrote to `reports/metrics/` and printed in the next cell.

## 1. Imports

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from seoul_air import config
from seoul_air.report import load_key_figures
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 140)
config.FIGURES.mkdir(parents=True, exist_ok=True)
config.METRICS.mkdir(parents=True, exist_ok=True)

## 2. The figures behind the findings

In [2]:
k = {nb: load_key_figures(nb) for nb in ["01", "02", "03", "04", "05"]}
month, bad, year = k["02"]["pm25_median_by_month"], k["02"]["bad_or_worse_pct_by_month"], k["02"]["pm25_mean_by_year"]
share, groups, forecast = k["03"]["variance_share_pct"], k["03"]["categories"], k["05"]
mae, selected = forecast["mae"], forecast["selected_model"]

facts = [
    ("Data", "Readings dropped for a faulty instrument status", f"{k['01']['dropped_pct']:.2f}%"),
    ("Data", "Days with no value for the city, of all days", f"{k['01']['gap_days']} of {k['01']['gap_days'] + k['01']['valid_city_days']:,}"),
    ("Level", "Mean daily PM2.5 in 2017 / 2018 / 2019 (ug/m3)", " / ".join(f"{year[y]:.1f}" for y in ["2017", "2018", "2019"])),
    ("Season", "Median daily PM2.5, March / September (ug/m3)", f"{month['3']:.1f} / {month['9']:.1f}"),
    ("Season", "Station-days graded Bad or worse, March / August", f"{bad['3']:.1f}% / {bad['8']:.1f}%"),
    ("Variation", "Share of the variation explained by the day / the calendar month / the district",
     f"{share['Date (each single day)']:.1f}% / {share['Calendar month (season)']:.1f}% / {share['District']:.1f}%"),
    ("Variation", "Lowest correlation of daily PM2.5 between two districts", f"{k['03']['min_pairwise_correlation']:.2f}"),
    ("Episodes", "City days Bad or worse, of valid days", f"{k['04']['bad_city_days']} of {k['04']['valid_city_days']:,}"),
    ("Episodes", "Episodes of 2 or more Bad days in a row", f"{k['04']['episodes_2plus_days']}"),
    ("Districts", "Stations with a consistent difference / not comparable, of 25",
     f"{groups.get('Above the rest of the city', 0) + groups.get('Below the rest of the city', 0)} / {groups['Not comparable']}"),
    ("Forecast", f"Error of the next-day forecast ({selected}) vs 'tomorrow = today' (ug/m3)", f"{mae[selected]:.2f} vs {mae['persistence']:.2f}"),
    ("Forecast", "Error reduction, with its 95% interval",
     f"{forecast['skill_vs_persistence_pct']:.1f}% ({forecast['skill_interval_pct'][0]:.1f}% to {forecast['skill_interval_pct'][1]:.1f}%)"),
    ("Forecast", "Bad days caught with the standard alert, model vs 'tomorrow = today'",
     f"{forecast['recall_grade_rule'][selected]:.0%} vs {forecast['recall_grade_rule']['persistence']:.0%}"),
    ("Forecast", "Bad days caught with the calibrated alert (threshold, ug/m3)",
     f"{forecast['recall_calibrated'][selected]:.0%} ({forecast['threshold'][selected]:g})"),
]
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(facts, columns=["topic", "figure", "value"]).set_index(["topic", "figure"])

value
topic     figure                                                                                                
Data      Readings dropped for a faulty instrument status                                                  2.81%
          Days with no value for the city, of all days                                               18 of 1,095
Level     Mean daily PM2.5 in 2017 / 2018 / 2019 (ug/m3)                                      24.6 / 22.8 / 24.9
Season    Median daily PM2.5, March / September (ug/m3)                                              34.4 / 10.7
          Station-days graded Bad or worse, March / August                                          48.2% / 0.4%
Variation Share of the variation explained by the day / the calendar month / the district   94.3% / 23.4% / 0.6%
          Lowest correlation of daily PM2.5 between two districts                                           0.90
Episodes  City days Bad or worse, of valid days                                                     186 of 1,077
          Episodes of 2 or more Bad days in a row                                                             48
Districts Stations with a consistent difference / not comparable, of 25                                    5 / 9
Forecast  Error of the next-day forecast (ridge) vs 'tomorrow = today' (ug/m3)                      7.97 vs 8.86
          Error reduction, with its 95% interval                                           10.0% (4.3% to 15.6%)
          Bad days caught with the standard alert, model vs 'tomorrow = today'                        53% vs 61%
          Bad days caught with the calibrated alert (threshold, ug/m3)                                  76% (29)

![Daily PM2.5 and PM10, 2017-2019](../reports/figures/02_daily_series.png)

## 3. Key findings

1. **The air was often unhealthy, and it did not clearly improve.** The average day had 24.6, 22.8 and 24.9 ug/m3 of PM2.5 in 2017, 2018 and 2019, well above the Korean annual standard of 15. Three years are too few to tell a trend from normal year-to-year change.
2. **Late winter and spring are the worst time of year.** The typical day in March had 34.4 ug/m3 against 10.7 in September; 48.2% of station-days in March were graded Bad or worse, against 0.4% in August.
3. **Bad air hits the whole city at once.** Even the two least similar districts have a daily correlation of 0.90. The day explains 94.3% of the variation, the calendar month 23.4% and the district only 0.6%: bad air comes in city-wide episodes, 48 of them lasting two days or more, rather than staying in particular districts.
4. **The data does not support a ranking of districts.** Stations were moved and re-equipped during these years, and their levels drift over time. Only 5 of the 25 stations keep a consistent difference from the rest of the city; 9 cannot be compared across the period.
5. **Tomorrow's level can be predicted a little better than by assuming it equals today.** The model's error is 10.0% smaller (7.97 against 8.86 ug/m3), and the 95% interval of that gain, 4.3% to 15.6%, does not include zero. As an alert it needs care: with the standard rule it catches 53% of Bad days against 61% for "tomorrow = today"; with a lower trigger chosen on 2018 (29 ug/m3) it catches 76%, although notebook 05 shows that this is not clearly more than the simple rule achieves with its own trigger. These 2019 figures come from a year that had already been looked at, so they may be slightly optimistic.

## 4. Recommendations

These follow from the patterns in the data; they are not tested interventions.

- **Plan protection by season and by episode.** Masks, indoor activities and public warnings matter most from December to March and during multi-day episodes, which this data shows to be city-wide.
- **Communicate at city level, day by day.** Because the day explains far more than the district, a daily forecast for the whole city is more useful to residents than a map ranking districts.
- **Use a lower, tested trigger for alerts.** A forecast that waits for the standard Bad level misses many Bad days; a trigger chosen on past data, at 29 ug/m3 here, catches more of them at the cost of more false alarms.
- **Do not compare districts without the station history.** Relocations and instrument changes can move a station's level; month-level maintenance records from the operator would be needed to separate them from real differences.

## 5. Limitations and next steps

- **Three years only (2017-2019).** The data stops before the PM seasonal management system (December 2019), so its effect cannot be assessed, and the trend is unknown.
- **No weather data.** Wind and rain shape daily PM2.5, but they are not in the dataset. Adding them is the main step to a better forecast.
- **One station per district**, and stations changed during the period (notebook 03).
- **2019 had been inspected** before the forecast protocol was fixed (notebook 05). A clean evaluation needs data from 2020 onward.
- **Hour labels:** the start-of-hour convention is verified against the Seoul Open API, but the archive of 2017-2019 is assumed to follow the convention seen in 2026 (notebook 01).

**Next steps** (see `docs/ANALYSIS_PLAN.md`): extend the data to 2015-2025 with weather, evaluate the seasonal management system, improve the 1-3 day forecast with weather forecasts, and build scenario forecasts for the coming 2-3 years.